# LeafTrack: train YOLO11-seg on Kaggle

Thin launcher: all logic lives in the `leaftrack` package.
In the notebook settings choose **Accelerator: GPU T4** and turn **Internet on**.

Kaggle cannot mount Google Drive, so runs are saved in `/kaggle/working` (kept as notebook output when you *Save Version*)
and the final weights should be pushed to the Hugging Face Hub. Archives are cached in `/kaggle/working` too.

In [ ]:
MODEL = "yolo11n"  # or "yolo11s"
HF_REPO = ""  # e.g. "your-user/leaftrack-yolo11n-seg" (needs HF_TOKEN in Add-ons → Secrets)
WORK = "/kaggle/working"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /kaggle/working
!test -d leaftrack && git -C leaftrack pull -q || git clone -q https://github.com/iamoyes1/leaftrack.git
%cd /kaggle/working/leaftrack
!pip install -q -e ".[train]"

In [ ]:
!python data/scripts/download_komatsuna.py --archives {WORK}/archives --out data/raw/komatsuna
!python data/scripts/convert_komatsuna.py
!python data/scripts/split_dataset.py

In [ ]:
!python train.py --config configs/train_{MODEL}.yaml --project {WORK}/runs --name {MODEL}

In [ ]:
!python evaluate.py --weights {WORK}/runs/{MODEL}/weights/best.pt --device 0 --out {WORK}/eval/{MODEL}
from IPython.display import Markdown, display

display(Markdown(open(f"{WORK}/eval/{MODEL}/results.md").read()))

In [ ]:
if HF_REPO:
    import os

    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    !python -m leaftrack.hub --weights {WORK}/runs/{MODEL}/weights/best.pt --repo {HF_REPO} --metrics {WORK}/eval/{MODEL}/metrics.json